In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

In [2]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 300)

In [3]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='jerin.steefan@rapido.bike'
)

In [4]:
## Parameter 
start_date = '20231019'
end_date = '20231019'

In [61]:
##clevertap_customer_orderactivity
clevertap_customer_orderactivity = f"""
WITH ao AS (
    
    SELECT
        DISTINCT
        yyyymmdd,
        quarter_hour,
        CAST(SUBSTR(quarter_hour,1,2) AS INT) hour,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') appOpen_userId,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') currentCity,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') screen_version,
        CAST(CAST(JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.CTSessionId') AS DECIMAL) AS VARCHAR) CTSessionId

    FROM 
        raw.clevertap_customer_events_master
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND profile_platform = 'Android'
        AND eventname = 'orderactivity'
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') IN ('M0.5', 'M0')
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.order_activity_source') = 'appOpen' 
    ),
    
    sa AS (
    
    SELECT 
        DISTINCT
        yyyymmdd,
        CAST(SUBSTR(quarter_hour,1,2) AS INT) hour,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') searchaddress_userId,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') currentCity,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') screen_version,
        CAST(CAST(JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.CTSessionId') AS DECIMAL) AS VARCHAR) CTSessionId 
    FROM 
        raw.clevertap_customer_events_master sa
    
    WHERE 
        yyyymmdd >= '{start_date}'
        AND yyyymmdd <= '{end_date}'
        AND eventname = 'searchaddress'
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') IN ('M0.5', 'M0')    
    ),
    
    fe AS (

    SELECT 
        fe.yyyymmdd,
        fe.user_id fe_userId,
        fe.city,
        fe.fare_estimate_id,
        -- service_name,
        fe.service_detail_id,
        sfe.session_id,
        CAST(SUBSTR(fe.quarter_hour,1,2) AS INT) hour
    FROM 
        pricing.fare_estimates_enriched fe
        
    JOIN sa
        ON sa.yyyymmdd = fe.yyyymmdd 
        AND sa.userId = fe.user_id
        AND sa.hour = CAST(SUBSTR(fe.quarter_hour,1,2) AS INT)

    LEFT JOIN hive.marketplace.session_fare_estimates sfe    
        ON sfe.yyyymmdd = fe.yyyymmdd
        AND sfe.fare_estimate_id = fe.fare_estimate_id

    WHERE 
        fe.yyyymmdd >= '{start_date}'
        AND fe.yyyymmdd <= '{end_date}'
        AND fe.city IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')

    GROUP BY 1,2,3,4,5,6,7
    ),

    rr AS (

    SELECT
        orderdate,
        city,
        service_name,
        service_detail_id,
        customer_id,
        u.id_array as fare_estimate_id,
        order_id,
        order_status,
        spd_fraud_flag
    FROM
        (
        SELECT   
            date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y%m%d') AS orderdate,
            city_name as city,
            service_obj_service_name as service_name,
            service_detail_id,
            customer_id,
            order_id,
            order_status,
            spd_fraud_flag,
            estimate_id,
            estimate_ids,
            cast(json_parse(estimate_ids) as array<varchar>) AS id_array
        FROM
            orders.order_logs_snapshot

        WHERE
            yyyymmdd >= '{start_date}'
            AND yyyymmdd <= '{end_date}'
            AND city_name IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
        ) as t
    CROSS JOIN UNNEST(t.id_array) AS u(id_array)

    JOIN 
        sa
        ON sa.userId = t.customer_id
    )

    SELECT 
        ao.yyyymmdd,
        ao.quarter_hour,
        ao.currentCity,
        ao.screen_version,
        ao.appOpen_userId,
        sa.searchaddress_userId,
        sa.hour,
        fe.fe_userId,
        fe.session_id,
        fe.fare_estimate_id,
        rr.customer_id rr_userId,
        rr.order_id,
        rr.order_status,
        rr.spd_fraud_flag
        
    FROM 
        ao
        
    LEFT JOIN
        sa
        ON ao.yyyymmdd = sa.yyyymmdd
        AND ao.userId = sa.userId
        AND ao.CTSessionId = sa.CTSessionId

    LEFT JOIN 
        fe
        ON sa.yyyymmdd = fe.yyyymmdd
        AND sa.userId = fe.fe_userId
        AND sa.hour = fe.hour
    
    LEFT JOIN 
        rr
        ON fe.yyyymmdd = rr.orderdate
        AND fe.fare_estimate_id = rr.fare_estimate_id
        AND fe.service_detail_id = rr.service_detail_id 
"""

In [62]:
df_clevertap_customer_orderactivity = pd.read_sql(clevertap_customer_orderactivity, connection)

In [63]:
df_clevertap_customer_orderactivity

,yyyymmdd,quarter_hour,currentCity,screen_version,appOpen_userId,searchaddress_userId,hour,fe_userId,session_id,fare_estimate_id,rr_userId,order_id,order_status,spd_fraud_flag
0,20231019,1515,Bangalore,M0.5,61448a61c1762ce445801950,61448a61c1762ce445801950,15.0,61448a61c1762ce445801950,20b5b402-e289-480f-9059-828d6d17c980,653102922797f2d54ad81a51,61448a61c1762ce445801950,6530fe4bcea7ca2ea409ed57,dropped,False
1,20231019,1130,Hyderabad,M0,6453dcceb7b8438c3af82b57,6453dcceb7b8438c3af82b57,11.0,6453dcceb7b8438c3af82b57,50a9d118-69c4-4758-a1bb-c73eb100a033,6530c664c031e495af8b5978,None,None,None,None
2,20231019,1215,Hyderabad,M0,615073f4fd38d3df7c88d1c7,615073f4fd38d3df7c88d1c7,12.0,615073f4fd38d3df7c88d1c7,55fefa79-aeda-4a47-879f-d3df63b88ea5,6530d299998f520d5796ead0,None,None,None,None
3,20231019,1600,Delhi,M0,652f7de2573069006263d003,652f7de2573069006263d003,16.0,652f7de2573069006263d003,c8dc071b-4a9d-4645-be4e-e6ec49bfef0c,65310bd68d0ef480b3a0821c,None,None,None,None
4,20231019,1615,Delhi,M0,652f7de2573069006263d003,652f7de2573069006263d003,16.0,652f7de2573069006263d003,c8dc071b-4a9d-4645-be4e-e6ec49bfef0c,65310bd68d0ef480b3a0821c,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503612,20231019,0815,Hyderabad,M0,5c7e6d6c5e042733c9c396f8,None,NaN,None,None,None,None,None,None,None
503613,20231019,2000,Delhi,M0,5bfb73e07b420539da3fc406,None,NaN,None,None,None,None,None,None,None
503614,20231019,1615,Hyderabad,M0.5,5d836a63486b0b21476de31d,None,NaN,None,None,None,None,None,None,None
503615,20231019,1130,Hyderabad,M0,628e563f1cd6c05b82766bd9,None,NaN,None,None,None,None,None,None,None


In [55]:
test = df_clevertap_customer_orderactivity \
.groupby(['currentCity', 'screen_version']) \
.agg(ao_customers = pd.NamedAgg('appOpen_userId' , 'nunique'),
     sa_customers = pd.NamedAgg('searchaddress_userId' , 'nunique'),
     fe_customers = pd.NamedAgg('fe_userId' , 'nunique'),
     rr_customers = pd.NamedAgg('rr_userId' , 'nunique'),
     fe_session = pd.NamedAgg('session_id' , 'nunique'),
     rr_session = ('session_id', 
                  lambda x: x[(df_clevertap_customer_orderactivity['order_id'].notna())
                              &
                              (df_clevertap_customer_orderactivity['order_id'] != '')
                             ] \
                  .nunique())
    ) \
.reset_index()
test

,currentCity,screen_version,ao_customers,sa_customers,fe_customers,rr_customers,fe_session,rr_session
0,Bangalore,M0,2992,2388,2344,1814,3287,2331
1,Bangalore,M0.5,2750,2254,2221,1735,3151,2237
2,Delhi,M0,3767,3108,3055,2343,4193,2882
3,Delhi,M0.5,3615,3064,3019,2369,4259,2948
4,Hyderabad,M0,7237,5697,5572,4038,7530,4866
5,Hyderabad,M0.5,6237,5034,4954,3759,6859,4611
6,Kolkata,M0,1055,823,800,512,1169,646
7,Kolkata,M0.5,948,758,742,477,1103,615


In [56]:
test['AO2RR'] = test['rr_customers']*100.0/test['ao_customers']
test['AO2FE'] = test['fe_customers']*100.0/test['ao_customers']
test['AO2SA'] = test['sa_customers']*100.0/test['ao_customers']
test['SA2FE'] = test['fe_customers']*100.0/test['sa_customers']
test['FE2RR'] = test['rr_customers']*100.0/test['fe_customers']
test['FE2RR session'] = test['rr_session']*100.0/test['fe_session']

In [60]:
test.round(2).to_clipboard(index=False)

In [34]:
clevertap_customer_searchaddress = f"""
SELECT 
    JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
    CAST(SUBSTR(quarter_hour,1,2) AS INT) quarter_hour,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') searchaddress_userId,
    JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') currentCity,
        JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') screen_version
FROM 
    raw.clevertap_customer_events_master
WHERE 
    yyyymmdd >= '{start_date}'
    AND yyyymmdd <= '{end_date}'
    AND eventname = 'searchaddress'
    AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
        AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') IN ('M0.5', 'M0')

GROUP BY 1,2,3,4,5
"""
df_clevertap_customer_searchaddress = pd.read_sql(clevertap_customer_searchaddress, connection)
df_clevertap_customer_searchaddress

,userId,quarter_hour,searchaddress_userId,currentCity,screen_version
0,5dcd0fe90856f146182b3f7a,22,5dcd0fe90856f146182b3f7a,Bangalore,M0.5
1,60c80920be7e7c33a91d9d92,14,60c80920be7e7c33a91d9d92,Delhi,M0
2,5bac594e1d0eb86e79943473,14,5bac594e1d0eb86e79943473,Delhi,M0
3,60730a74de2a0a9e35259fed,13,60730a74de2a0a9e35259fed,Hyderabad,M0.5
4,5d492f8755fbf50d45e5b269,13,5d492f8755fbf50d45e5b269,Delhi,M0
...,...,...,...,...,...
41098,6443c7d4a9cc6f4a4bfd1483,10,6443c7d4a9cc6f4a4bfd1483,Hyderabad,M0
41099,61aa2073d3152a64bfc397df,10,61aa2073d3152a64bfc397df,Hyderabad,M0.5
41100,5e4ba60545a45901d5b5947f,9,5e4ba60545a45901d5b5947f,Bangalore,M0.5
41101,624fe4389b264ea4f8d06101,9,624fe4389b264ea4f8d06101,Hyderabad,M0.5


In [25]:
df_clevertap_customer_orderactivity[df_clevertap_customer_orderactivity['CTSessionId'] == '1697703162']

,yyyymmdd,userId,appOpen_userId,currentCity,screen_version,CTSessionId
843,20231019,6530d7f6bb1bff2c11020aa7,6530d7f6bb1bff2c11020aa7,Kolkata,M0,1697703162
11851,20231019,64c5ecf21e3e963b65bd7d02,64c5ecf21e3e963b65bd7d02,Hyderabad,M0,1697703162


In [26]:
df_clevertap_customer_searchaddress[df_clevertap_customer_searchaddress['CTSessionId'] == '1697703162']

,userId,searchaddress_userId,currentCity,screen_version,CTSessionId
45154,64c5ecf21e3e963b65bd7d02,64c5ecf21e3e963b65bd7d02,Hyderabad,M0,1697703162


In [73]:
# Set your start and end dates
start_date = datetime.strptime('20230101', '%Y%m%d')
end_date = datetime.strptime('20230104', '%Y%m%d')

# Initialize an empty DataFrame to store the results
result_df = pd.DataFrame()

# Loop through each day within the date range
current_date = start_date
while current_date <= end_date:
    # Convert the current date to the required format
    current_date_str = current_date.strftime('%Y%m%d')

    # Build the query with the current date
    query = f"""
        SELECT 
            *
        FROM
            raw.clevertap_customer_postorderstatus_requesting
        WHERE 
            yyyymmdd = '{current_date_str}'
        LIMIT 5
    """
    
    # Execute the query and append the results to the DataFrame
    print('reading', {current_date_str})
    day_data = pd.read_sql(query, connection)
    result_df = pd.concat([result_df, day_data], ignore_index=True)
    print('done')

    # Move to the next day
    current_date += timedelta(days=1)

reading {'20230101'}
done
reading {'20230102'}
done
reading {'20230103'}
done
reading {'20230104'}
done


In [72]:
result_df

,eventname,yyyymmdd,hhmmss,quarter_hour,epoch,created_ist,deviceInfo_appVersion,deviceInfo_browser,deviceInfo_dimensions_height,deviceInfo_dimensions_unit,deviceInfo_dimensions_width,deviceInfo_dpi,deviceInfo_make,deviceInfo_model,deviceInfo_osVersion,deviceInfo_sdkVersion,profile_all_identities,profile_email,profile_identity,profile_name,profile_phone,profile_platform,eventProps_area,eventProps_currentCity,eventProps_currentCluster,eventProps_epoch,eventProps_hhmmss,eventProps_latitude,eventProps_locationAccuracy,eventProps_longitude,eventProps_occurred_date,eventProps_yyyymmdd,eventProps_CTAppVersion,eventProps_CTLatitude,eventProps_CTLongitude,eventProps_CTSource,eventProps_CTSessionId,deviceInfo_data,profile_data,eventProps_data
0,postorderstatus_requesting,20230101,044751,0445,1672528671000,2023-01-01 04:47:51.000,7.14.0,MobileApp,115,mm,57,320,realme,RMX2027,11,40601,"[""6384d79179f9eaf4a2ab9b63""]",None,6384d79179f9eaf4a2ab9b63,NITESH VERMA,918115215568,Android,----,Lucknow,Charbagh,1672528671146,044751,26.837166,8.031,80.910225,Sun Jan 01 2023 04:47:51 am,20230101,7.14.0,26.837166,80.910225,Mobile,1.672526174E9,"{""appVersion"":""7.14.0"",""browser"":""MobileApp"",""...","{""all_identities"":[""6384d79179f9eaf4a2ab9b63""]...","{""area"":""----"",""cleverTapId"":""__g15777b90f3824..."
1,postorderstatus_requesting,20230101,044825,0445,1672528705000,2023-01-01 04:48:25.000,7.14.0,MobileApp,129,mm,49,420,samsung,SM-F415F,12,40601,"[""5a006e555422e0655cb0591a"",""selvavindhan4@gma...",selvavindhan4@gmail.com,5a006e555422e0655cb0591a,G V N Selvavindhan,919632409473,Android,Azad Nagar,Mumbai,MUM_SOBO_Fort,1672528705181,044825,18.925896,21.539,72.82944,Sun Jan 01 2023 04:48:25 am,20230101,7.14.0,18.925867,72.82943,Mobile,1.672527914E9,"{""appVersion"":""7.14.0"",""browser"":""MobileApp"",""...","{""all_identities"":[""5a006e555422e0655cb0591a"",...","{""area"":""Azad Nagar"",""cleverTapId"":""__g7145ac0..."
2,postorderstatus_requesting,20230101,051112,0500,1672530072000,2023-01-01 05:11:12.000,7.13.0,MobileApp,112,mm,57,320,tecno,KG5p,12,40601,"[""632a950b9408523597be0f73""]",None,632a950b9408523597be0f73,padma,919705503790,Android,Dammaiguda,Hyderabad,AOC Center,1672530072099,051112,17.507742,3.9,78.58992,Sun Jan 01 2023 05:11:12 am,20230101,7.13.0,17.507742,78.58992,Mobile,1.6725296E9,"{""appVersion"":""7.13.0"",""browser"":""MobileApp"",""...","{""all_identities"":[""632a950b9408523597be0f73""]...","{""area"":""Dammaiguda"",""cleverTapId"":""__g3397f09..."
3,postorderstatus_requesting,20230101,054224,0530,1672531944000,2023-01-01 05:42:24.000,7.13.0,MobileApp,111,mm,57,480,oppo,CPH1945,11,40601,"[""62a45fdb69f12b8703ef0ad7""]",None,62a45fdb69f12b8703ef0ad7,Solipuram chandrasekhar reddy,919550050011,Android,----,Hyderabad,Nallagandla,1672531944331,054224,17.4794,1.766,78.30945,Sun Jan 01 2023 05:42:24 am,20230101,7.13.0,17.4794,78.30945,Mobile,1.672530602E9,"{""appVersion"":""7.13.0"",""browser"":""MobileApp"",""...","{""all_identities"":[""62a45fdb69f12b8703ef0ad7""]...","{""area"":""----"",""cleverTapId"":""__g5e46ab3111914..."
4,postorderstatus_requesting,20230101,062219,0615,1672534339000,2023-01-01 06:22:19.000,7.14.0,MobileApp,137,mm,68,320,realme,RMX2030,10,40601,"[""61946bdaae4f7377b41be111"",""kumarbudhdev17@gm...",kumarbudhdev17@gmail.com,61946bdaae4f7377b41be111,BUDHDEV KUMAR,919123104728,Android,Murphy Town,Bangalore,BLR_Indiranagar,1672534339425,062219,12.983053,43.114,77.63777,Sun Jan 01 2023 06:22:19 am,20230101,7.14.0,12.983053,77.63777,Mobile,1.672533776E9,"{""appVersion"":""7.14.0"",""browser"":""MobileApp"",""...","{""all_identities"":[""61946bdaae4f7377b41be111"",...","{""area"":""Murphy Town"",""cleverTapId"":""__gd89de1..."
5,postorderstatus_requesting,20230102,001510,0015,1672598710000,2023-01-02 00:15:10.000,7.14.0,MobileApp,114,mm,57,320,realme,RMX2103,11,40601,"[""63a5b9f8364fef55bba6e55e""]",None,63a5b9f8364fef55bba6e55e,guru,918374042179,Android,Film Nagar,Hyderabad,Manikonda,1672598710420,001510,

In [ ]:
df_data_dump = pd.DataFrame()

current_date = start_date_range
while current_date <= end_date_range:
    
    current_date_str = current_date.strftime('%Y%m%d')

    query = f"""
        WITH ao AS (
    
        SELECT
            DISTINCT
            yyyymmdd,
            quarter_hour,
            CAST(SUBSTR(quarter_hour,1,2) AS INT) hour,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') appOpen_userId,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') currentCity,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') screen_version,
            CAST(CAST(JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.CTSessionId') AS DECIMAL) AS VARCHAR) CTSessionId

        FROM 
            raw.clevertap_customer_events_master
        WHERE 
            yyyymmdd = '{current_date_str}'
            AND profile_platform = 'Android'
            AND eventname = 'orderactivity'
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') IN ('M0.5', 'M0')
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.order_activity_source') = 'appOpen' 
        ),

        sa AS (

        SELECT 
            DISTINCT
            yyyymmdd,
            CAST(SUBSTR(quarter_hour,1,2) AS INT) hour,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') userId,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.userId') searchaddress_userId,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') currentCity,
            JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') screen_version,
            CAST(CAST(JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.CTSessionId') AS DECIMAL) AS VARCHAR) CTSessionId 
        FROM 
            raw.clevertap_customer_events_master sa

        WHERE 
            yyyymmdd = '{current_date_str}'
            AND eventname = 'searchaddress'
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.currentCity') IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
            AND JSON_EXTRACT_SCALAR(eventProps, '$.eventProps.screen_version') IN ('M0.5', 'M0')    
        ),

        fe AS (

        SELECT 
            fe.yyyymmdd,
            fe.user_id fe_userId,
            fe.city,
            fe.fare_estimate_id,
            -- service_name,
            fe.service_detail_id,
            sfe.session_id,
            CAST(SUBSTR(fe.quarter_hour,1,2) AS INT) hour
        FROM 
            pricing.fare_estimates_enriched fe

        JOIN sa
            ON sa.yyyymmdd = fe.yyyymmdd 
            AND sa.userId = fe.user_id
            AND sa.hour = CAST(SUBSTR(fe.quarter_hour,1,2) AS INT)

        LEFT JOIN hive.marketplace.session_fare_estimates sfe    
            ON sfe.yyyymmdd = fe.yyyymmdd
            AND sfe.fare_estimate_id = fe.fare_estimate_id

        WHERE 
            fe.yyyymmdd = '{current_date_str}'
            AND fe.city IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')

        GROUP BY 1,2,3,4,5,6,7
        ),

        rr AS (

        SELECT
            orderdate,
            city,
            service_name,
            service_detail_id,
            customer_id,
            u.id_array as fare_estimate_id,
            order_id,
            order_status,
            spd_fraud_flag
        FROM
            (
            SELECT   
                date_format(from_unixtime(epoch / 1000, 'Asia/Kolkata'), '%Y%m%d') AS orderdate,
                city_name as city,
                service_obj_service_name as service_name,
                service_detail_id,
                customer_id,
                order_id,
                order_status,
                spd_fraud_flag,
                estimate_id,
                estimate_ids,
                cast(json_parse(estimate_ids) as array<varchar>) AS id_array
            FROM
                orders.order_logs_snapshot

            WHERE
                yyyymmdd = '{current_date_str}'
                AND city_name IN ('Hyderabad', 'Delhi', 'Kolkata', 'Bangalore')
            ) as t
        CROSS JOIN UNNEST(t.id_array) AS u(id_array)

        JOIN 
            sa
            ON sa.userId = t.customer_id
        )

        SELECT 
            ao.yyyymmdd,
            ao.quarter_hour,
            ao.currentCity,
            ao.screen_version,
            ao.appOpen_userId,
            sa.searchaddress_userId,
            sa.hour,
            fe.fe_userId,
            fe.session_id,
            fe.fare_estimate_id,
            rr.customer_id rr_userId,
            rr.order_id,
            rr.order_status,
            rr.spd_fraud_flag

        FROM 
            ao

        LEFT JOIN
            sa
            ON ao.yyyymmdd = sa.yyyymmdd
            AND ao.userId = sa.userId
            AND ao.CTSessionId = sa.CTSessionId

        LEFT JOIN 
            fe
            ON sa.yyyymmdd = fe.yyyymmdd
            AND sa.userId = fe.fe_userId
            AND sa.hour = fe.hour

        LEFT JOIN 
            rr
            ON fe.yyyymmdd = rr.orderdate
            AND fe.fare_estimate_id = rr.fare_estimate_id
            AND fe.service_detail_id = rr.service_detail_id 

    """
    
    # Execute the query and append the results to the DataFrame
    print('Reading ->', current_date_str)
    day_data = pd.read_sql(query, connection)
    df_data_dump = pd.concat([df_data_dump, day_data], ignore_index=True)

    # Move to the next day
    current_date += timedelta(days=1)

df_data_dump.to_csv(data_path + 
                    'df_data_dump_{}_{}.csv'.format(start_date,end_date),
                    index=False)